In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

import matplotlib.pyplot as plt
import importlib
import numpy as np

from typing import TYPE_CHECKING, Callable, Union, Optional

from VariablesClass import VariablesClass
from StructureClass import  StructureClass
from StateClass import StateClass
from EquilibriumClass import EquilibriumClass
from SupervisorClass import SupervisorClass
from config import CFG

import plot_funcs, colors, helpers_builders, learning_funcs, file_funcs, numerical_experiments

## Average training time

In [ ]:
folder = "Training\\Apr23randomPosAfterFroceExplode"
average_t = file_funcs.average_successful_train_time(folder = folder, thresh=1e-6) 
print("average_t = ", average_t)

## Non abelianity check

In [ ]:
import config
importlib.reload(config)
from config import CFG

# Non-abelian buckle check: inputs
importlib.reload(numerical_experiments)

Strctr = StructureClass(CFG, update_scheme=CFG.Train.update_scheme)
Variabs = VariablesClass(Strctr, CFG)
Sprvsr = SupervisorClass(Strctr, CFG, supress_prints=False)
Sprvsr.create_dataset(Strctr, CFG, CFG.Train.dataset_sampling, tip_pos=None, tip_angle=None)

init_buckle = helpers_builders._initiate_buckle(
    CFG.Strctr.H,
    CFG.Strctr.S,
    buckle_pattern=CFG.Train.init_buckle_pattern,
)

flat_tip_pos = np.array([Strctr.edges * Strctr.L, 0.0])
flat_tip_angle = 0.0

# Edit these values for the check.
initial_tip_pos = np.array([0.82 * Strctr.edges * Strctr.L, -0.03])
initial_tip_angle = -0.05
final_tip_pos = np.array([0.5 * Strctr.edges * Strctr.L, -0.3 * Strctr.edges * Strctr.L])
final_tip_angle = -np.pi/2
Eq_iterations = 4

print("flat pose:", flat_tip_pos, flat_tip_angle)
print("initial pose:", initial_tip_pos, initial_tip_angle)
print("final pose:", final_tip_pos, final_tip_angle)
print("initial buckle:", np.asarray(init_buckle, dtype=int).reshape(-1))

In [ ]:
# Non-abelian buckle check: warm start, then compare operation order
compress_to_tip_position = getattr(
    numerical_experiments,
    "compress_to_tip_position",
    numerical_experiments.compress_to_tip_pos,
)

def buckle_tuple(buckle_arr):
    return tuple(np.asarray(buckle_arr, dtype=int).reshape(-1))

def run_leg(label, buckle, tip_pos_i, tip_angle_i, tip_pos_f, tip_angle_f, init_pos):
    print(f"\n{label}")
    State, pos_in_t, force_in_t = compress_to_tip_position(
        Strctr,
        Variabs,
        Sprvsr,
        CFG,
        np.asarray(buckle, dtype=int).copy(),
        np.asarray(tip_pos_i, dtype=float),
        float(tip_angle_i),
        np.asarray(tip_pos_f, dtype=float),
        float(tip_angle_f),
        int(Eq_iterations),
        init_pos=None if init_pos is None else np.asarray(init_pos, dtype=float).copy(),
    )
    print("buckle:", buckle_tuple(State.buckle_arr))
    return State, pos_in_t, force_in_t

State_initial, pos_warm, force_warm = run_leg(
    "warm start: flat -> initial pose",
    init_buckle,
    flat_tip_pos,
    flat_tip_angle,
    initial_tip_pos,
    initial_tip_angle,
    init_pos=None,
)
initial_pos_arr = State_initial.pos_arr.copy()
initial_buckle_arr = State_initial.buckle_arr.copy()

State_pos, pos_pos, force_pos = run_leg(
    "path A, leg 1: move position first",
    initial_buckle_arr,
    initial_tip_pos,
    initial_tip_angle,
    final_tip_pos,
    initial_tip_angle,
    init_pos=initial_pos_arr,
)
State_pos_angle, pos_pos_angle, force_pos_angle = run_leg(
    "path A, leg 2: then move angle",
    State_pos.buckle_arr,
    final_tip_pos,
    initial_tip_angle,
    final_tip_pos,
    final_tip_angle,
    init_pos=State_pos.pos_arr,
)

State_angle, pos_angle, force_angle = run_leg(
    "path B, leg 1: move angle first",
    initial_buckle_arr,
    initial_tip_pos,
    initial_tip_angle,
    initial_tip_pos,
    final_tip_angle,
    init_pos=initial_pos_arr,
)
State_angle_pos, pos_angle_pos, force_angle_pos = run_leg(
    "path B, leg 2: then move position",
    State_angle.buckle_arr,
    initial_tip_pos,
    final_tip_angle,
    final_tip_pos,
    final_tip_angle,
    init_pos=State_angle.pos_arr,
)

buckle_pos_then_angle = buckle_tuple(State_pos_angle.buckle_arr)
buckle_angle_then_pos = buckle_tuple(State_angle_pos.buckle_arr)
non_abelian = buckle_pos_then_angle != buckle_angle_then_pos

print("\nFinal buckle after position -> angle:", buckle_pos_then_angle)
print("Final buckle after angle -> position:", buckle_angle_then_pos)
print("Non-abelian in buckle space:", non_abelian)